In [7]:
import requests
import pandas as pd
import duckdb
import json
from pathlib import Path
import re
from numpy import char

In [4]:
BASE = "https://data360api.worldbank.org"

def search_indicators(keyword="*", top=20):
    """Buscar indicadores por tema."""
    resp = requests.post(f"{BASE}/data360/searchv2", json={
        "count": False,
        # "filter": f"series_description/topics/any(t: t/name eq '{topic}') and type eq 'indicator'",
        "select": "series_description/idno, series_description/name, series_description/database_id",
        "search": keyword,
        "top": top
    })
    return resp.json()

def get_data_to_parquet(
    database_id,
    indicator,
    countries=None,
    time_from=None,
    time_to=None,
    output_path="../data/raw/data.parquet"
):
    """Fetch paginated API data and stream directly to a parquet file."""
    if database_id is None: return
    
    params = {"DATABASE_ID": database_id, "INDICATOR": indicator}
    if countries:
        params["REF_AREA"] = countries
    if time_from:
        params["timePeriodFrom"] = time_from
    if time_to:
        params["timePeriodTo"] = time_to

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    con = duckdb.connect()
    con.execute("CREATE TABLE temp (data JSON)")  # staging table

    skip = 0
    table_created = False

    while True:
        params["skip"] = skip
        resp = requests.get(f"{BASE}/data360/data", params=params)
        resp.raise_for_status()
        values = resp.json().get("value", [])

        if not values:
            break

        # Write batch to a temp JSON file DuckDB can read
        tmp_file = Path("/tmp/_batch.json")
        tmp_file.write_text(json.dumps(values))

        if not table_created:
            con.execute(f"""
                CREATE TABLE results AS
                SELECT * FROM read_json_auto('{tmp_file}')
            """)
            table_created = True
        else:
            con.execute(f"""
                INSERT INTO results
                SELECT * FROM read_json_auto('{tmp_file}')
            """)

        if len(values) < 1000:
            break

        skip += 1000

    # Write everything to parquet in one shot
    if table_created:
        con.execute(f"""
            COPY results TO '{output_path}' (FORMAT PARQUET)
        """)
        print(f"Saved to {output_path}")
    else:
        print("No data returned.")

    con.close()


In [8]:
indicators = [
    #'internet','wealth','education','corruption','freedom','development','commerce','trade','corruption','politic', 'fdi','urbanization','human rights',
    'civil society','rule of law', 'press freedom','governance']

for indicator in indicators:
    result = search_indicators(indicator)
    for i in result['value']:
        meta = i['series_description']
        name = "_".join([x.lower() for x in meta['name'].split(' ') if char.isalnum(x) and x not in ['the','of','at','a',]])
        
        output_file = f'../data/raw/{indicator}/{name}.parquet'
        if Path(output_file).is_file():
            continue
        get_data_to_parquet(meta['database_id'],meta['idno'],output_path=output_file)
    

Saved to ../data/raw/civil society/nations_in_civil_society.parquet
Saved to ../data/raw/civil society/bertelsmann_transformation_civil_society_traditions.parquet
Saved to ../data/raw/civil society/democracy_civil_liberties_score.parquet
Saved to ../data/raw/civil society/criminal_penalties_or_civil_remedies_exist_for_sexual_harassment_in_employment.parquet
Saved to ../data/raw/civil society/in_civil_servants_are_required_to_report_cases_alleged.parquet
Saved to ../data/raw/civil society/sustainable_governance_civil_rights.parquet
Saved to ../data/raw/civil society/in_civil_work_is_not_compromised_by_political.parquet
Saved to ../data/raw/civil society/freedom_civil_liberties_score_score.parquet
Saved to ../data/raw/civil society/freedom_in_civil_liberties_scores.parquet
Saved to ../data/raw/rule of law/wjp_rule_law_due_process_law_and_rights_accused.parquet
Saved to ../data/raw/rule of law/wjp_rule_law_factor_criminal_justice.parquet
Saved to ../data/raw/rule of law/wjp_rule_law_trans